# exp000 — ink_9um on a PHerc 0841 / w00 crop (Kaggle, T4)

**Before running**
- Accelerator: **GPU T4** (not P100: Pascal is likely unsupported by torch 2.12). Internet: **on**.
- Keep this notebook **private** and do not publish its outputs (data terms: see `LICENSES.md`).
- Config: `configs/exp000_841w00_ink9um_s42.yaml`. Downloads: ~138 MB checkpoint, ~35–40 MB of
  volume/label chunks; the uv environment (torch etc.) is a few GB inside the Kaggle VM.

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
python --version; df -h /kaggle/working | tail -1
git clone -q --depth 1 https://github.com/andreluizpedroso/Vesuvius-Challenge.git /kaggle/working/harness
git -C /kaggle/working/harness log -1 --format='harness %H'

In [ ]:
# Read the experiment config and export everything the bash cells need.
import os, yaml, pathlib
EXP = "configs/exp000_841w00_ink9um_s42.yaml"
cfg = yaml.safe_load((pathlib.Path("/kaggle/working/harness") / EXP).read_text())
os.environ.update({
    "EXP_NAME": cfg["experiment"],
    "EXP_CFG": f"/kaggle/working/harness/{EXP}",
    "VILLA_SHA": cfg["upstream"]["villa_commit"],
    "PY_VER": cfg["upstream"]["python"],
    "UV_SYNC_ARGS": " ".join(cfg["upstream"]["uv_sync_args"]),
    "CK_REPO": cfg["checkpoint"]["hf_repo"],
    "CK_REV": cfg["checkpoint"]["hf_revision"],
    "CK_FILE": cfg["checkpoint"]["file"],
    "INFER_MODULE": cfg["inference"]["module"],
    "INFER_ARGS": " ".join(cfg["inference"]["args"]),
    "VES": "/kaggle/working/villa/vesuvius",
    "CKPT": f"/kaggle/working/checkpoints/ink_9um/{cfg['checkpoint']['file']}",
    "DATA": "/kaggle/working/data/841_w00_crop",
    "OUT": f"/kaggle/working/outputs/{cfg['experiment']}",
})
print({k: os.environ[k] for k in ["EXP_NAME", "VILLA_SHA", "CK_FILE", "INFER_ARGS"]})

## 1. Upstream code (villa, pinned commit, sparse checkout) + environment
Only the folders `uv sync` needs. `volume-cartographer` (C++ bindings) is checked out for the lock but **not installed**.

In [ ]:
%%bash
set -e
git init -q /kaggle/working/villa && cd /kaggle/working/villa
git remote add origin https://github.com/ScrollPrize/villa.git
git sparse-checkout set vesuvius volume-cartographer segmentation/models/batchgeneratorsv2 segmentation/models/arch/nnunet
git fetch -q --depth 1 --filter=blob:none origin "$VILLA_SHA"
git checkout -q FETCH_HEAD && git log -1 --format='villa %H %cd'

In [ ]:
%%bash
set -e
pip install -q uv==0.11.24
uv python install "$PY_VER"
cd "$VES" && uv sync --python "$PY_VER" $UV_SYNC_ARGS 2>&1 | tail -20
uv run --no-sync python -c "import torch, sys; print(sys.version.split()[0], torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-', torch.cuda.get_arch_list())"

## 2. Data terms — **uncomment and run this cell yourself** after reading the printed terms
The harness never runs it automatically.

In [ ]:
%%bash
# cd "$VES" && uv run --no-sync vesuvius.accept_terms --yes

## 3. Checkpoint (pinned HF revision)

In [ ]:
%%bash
set -e
uvx --from huggingface_hub==1.22.0 hf download "$CK_REPO" "$CK_FILE" --revision "$CK_REV" --local-dir /kaggle/working/checkpoints/ink_9um
ls -la "$CKPT" && sha256sum "$CKPT"

## 4. Crop (lazy chunk reads; writes `crop.json` with bbox, coverage, level check, hashes)

In [ ]:
%%bash
set -e
cd "$VES" && uv run --no-sync python /kaggle/working/harness/scripts/fetch_crop.py --config "$EXP_CFG" --out "$DATA"

## 5. Inference

In [ ]:
%%bash
set -eo pipefail
mkdir -p "$OUT"
cd "$VES" && uv run --no-sync python -m "$INFER_MODULE" "$DATA/volume.zarr" "$CKPT" "$OUT/pred.tif" $INFER_ARGS 2>&1 | tee "$OUT/infer.log" | tail -40
ls -la "$OUT"

## 6. Run record (numbers only; no images displayed or saved for publication)

In [ ]:
%%bash
cd "$VES" && uv run --no-sync python - <<'EOF'
import glob, hashlib, json, os
import tifffile, torch
out = os.environ["OUT"]
rec = {"torch": torch.__version__, "gpu": torch.cuda.get_device_name(0), "preds": {}}
for p in sorted(glob.glob(out + "/*.tif")):
    a = tifffile.imread(p)
    rec["preds"][os.path.basename(p)] = {
        "shape": list(a.shape), "dtype": str(a.dtype), "min": int(a.min()), "max": int(a.max()),
        "mean": float(a.mean()), "sha256": hashlib.sha256(open(p, "rb").read()).hexdigest()}
json.dump(rec, open(out + "/run.json", "w"), indent=2)
print(json.dumps(rec, indent=2))
EOF
cp "$DATA/crop.json" "$OUT/crop.json"

## 7. Bring results back
Download `outputs/<exp>/crop.json`, `run.json` and `infer.log` (Kaggle → Output panel) and paste them in the chat,
or save them locally under `outputs/` (git-ignored). Copy `crop.json → selection.bbox_yx` into the config's `crop.bbox_yx`.